In [0]:
-- silver 
create or replace table workspace.redfin_silver.redfin_user_growth as
select
    day,
    nullif(trim(environment), '') as environment,
    device,
    medium,
    business_market,
    sessions,
    case
        when environment is null or trim(environment) = '' then true
        else false
    end as is_missing_environment,
    case
        when sessions = 0 then true
        else false
    end as is_zero_sessions,
    case
        when medium in ('social', 'internal') then true
        else false
    end as is_excluded_medium
from workspace.redfin_bronze.redfin_user_growth 

In [0]:
select
    count(*) as total_records,
    sum(sessions) as total_sessions,
    sum(case when is_missing_environment then 1 else 0 end) as missing_environment_records,
    sum(case when is_zero_sessions then 1 else 0 end) as zero_session_records,
    sum(case when is_excluded_medium then 1 else 0 end) as excluded_medium_records,
    sum(case when medium = 'social' then 1 else 0 end) as social_records,
    sum(case when medium = 'internal' then 1 else 0 end) as internal_records
from workspace.redfin_silver.redfin_user_growth

In [0]:
-- gold 
create or replace table workspace.redfin_gold.fact_daily_traffic as
select
    d.calendar_date as day,
    d.year,
    d.month,
    d.year_month,
    dayofweek(d.calendar_date) as day_of_week,
    s.environment,
    s.device,
    s.medium,
    s.business_market,
    s.sessions,
    s.is_missing_environment,
    s.is_zero_sessions,
    d.is_missing_date,
    d.is_future_date,
    d.is_reporting_cutoff_boundary
from workspace.redfin_gold.dim_date d
left join workspace.redfin_silver.redfin_user_growth s
    on d.calendar_date = s.day
    and s.is_excluded_medium = false

In [0]:
select
    count(*) as total_records,
    sum(sessions) as total_sessions,
    count(distinct day) as observed_days,
    sum(case when is_missing_environment then 1 else 0 end) as missing_environment_records,
    sum(case when is_zero_sessions then 1 else 0 end) as zero_session_records,
    sum(case when is_missing_date then 1 else 0 end) as missing_date_records,
    sum(case when is_future_date then 1 else 0 end) as future_date_records,
    sum(case when is_reporting_cutoff_boundary then 1 else 0 end) as cutoff_boundary_records
from workspace.redfin_gold.fact_daily_traffic

In [0]:
select * 
from workspace.redfin_gold.fact_daily_traffic
where medium in ('internal', 'social') 

In [0]:
-- traffic coverage 
select
    count(distinct case when sessions is not null then day end) as days_with_traffic,
    count(distinct case when is_missing_date then day end) as missing_historical_days,
    count(distinct case when is_future_date then day end) as future_days,
    count(distinct case when sessions is null then day end) as days_without_traffic_records
from workspace.redfin_gold.fact_daily_traffic

In [0]:
-- daily report view: 
create or replace view workspace.redfin_gold.daily_traffic as
with daily_sessions as (
    select
        day,
        sum(sessions) as total_sessions
    from workspace.redfin_gold.fact_daily_traffic
    group by day
)
select
    d.calendar_date as day,
    d.year,
    d.month,
    d.year_month,
    daily_sessions.total_sessions,
    d.is_missing_date,
    d.is_future_date,
    d.is_reporting_cutoff_boundary
from workspace.redfin_gold.dim_date d
left join daily_sessions
    on d.calendar_date = daily_sessions.day

In [0]:
describe extended workspace.redfin_gold.daily_traffic

In [0]:
select
    count(*) as total_calendar_days,
    count(total_sessions) as days_with_traffic,
    sum(case when is_missing_date then 1 else 0 end) as missing_historical_days,
    sum(case when is_future_date then 1 else 0 end) as future_days,
    sum(total_sessions) as total_sessions
from workspace.redfin_gold.daily_traffic
